In [ ]:
import json, sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
%matplotlib inline

repo = Path("..").resolve()
sys.path.insert(0, str(repo))

from scripts.common import collect_images, resolve_path, subsample_images
from src.evaluation.gt_assign import build_gt_from_images

COMMON = dict(seed=153, random_sample=False, box_color="lime", figsize=(12, 8), dpi=120)

In [ ]:
def render_entries(entries, out_dir, *, box_color, figsize, dpi, show_category):

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    saved = 0
    for i, entry in enumerate(entries):
        path = resolve_path(entry["image_path"])
        if not path.is_file():
            print(f"[skip] missing image: {path}")
            continue
        boxes  = entry["boxes_xyxy"]
        scores = entry.get("scores", [])
        cats   = entry.get("gt_categories", []) if show_category else []
        img    = np.asarray(Image.open(path).convert("RGB"))

        fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
        ax.imshow(img)
        for j, (x0, y0, x1, y1) in enumerate(boxes):
            ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0,
                                       fill=False, edgecolor=box_color, linewidth=2))
            sc  = float(scores[j]) if j < len(scores) else 0.0
            cat = str(cats[j]) if (show_category and j < len(cats) and cats[j]) else ""
            ax.text(x0, y0, f"{cat} {sc:.2f}".strip(),
                    fontsize=8, color="black",
                    bbox=dict(facecolor="yellow", alpha=0.5, pad=1))
        ax.set_title(path.name)
        ax.axis("off")
        fig.tight_layout()
        fig.savefig(out_dir / f"{i:04d}_{path.stem}.png")
        plt.close(fig)
        saved += 1
    return saved


def visualize_detections_json(det_path, out_dir, *, max_images=0, **kw):
    data = json.loads(Path(det_path).read_text())
    data = subsample_images(data, max_images, seed=kw["seed"], shuffle=kw["random_sample"])
    n = render_entries(
        data, out_dir,
        box_color=kw["box_color"], figsize=kw["figsize"], dpi=kw["dpi"],
        show_category=bool(data and data[0].get("gt_categories")),
    )
    print(f"Saved {n} PNGs under {Path(out_dir)}")


def visualize_coco_gt(ann_paths, images_dir, out_dir, *, min_box_side=2.0, max_images=0, **kw):
    folder = resolve_path(images_dir)
    image_paths = collect_images(folder)
    if not image_paths:
        raise FileNotFoundError(f"No images under {folder}")
    image_paths = subsample_images(image_paths, max_images,
                                   seed=kw["seed"], shuffle=kw["random_sample"])

    ann_paths = [resolve_path(p) for p in ann_paths]
    for p in ann_paths:
        if not p.is_file():
            raise FileNotFoundError(f"Annotation file not found: {p}")

    entries = build_gt_from_images(image_paths, ann_paths, min_box_side=min_box_side)
    n = render_entries(
        entries, out_dir,
        box_color=kw["box_color"], figsize=kw["figsize"], dpi=kw["dpi"],
        show_category=True,
    )
    print(f"Saved {n} PNGs under {Path(out_dir)} (COCO GT / encoding-style paths)")

In [ ]:
# DDETR: visualize detections.json
out_dir = repo / "outputs/notebook/viz_ddetr"

print("[visualize] mode: DDETR detections JSON")
visualize_detections_json(
    resolve_path(repo / "outputs/notebook/detect/bboxes/detections.json"),
    out_dir,
    max_images=0,   # 0 = all
    **COMMON,
)

In [ ]:
# COCO GT: visualize from annotation JSON
out_dir = repo / "outputs/notebook/viz_gt"

print("[visualize] mode: COCO GT")
visualize_coco_gt(
    [resolve_path(repo / "data/OWDETR/VOC2007/Annotations/instances_train2017.json")],
    resolve_path(repo / "data/OWDETR/VOC2007/JPEGImages"),
    out_dir,
    min_box_side=2.0,
    max_images=5,   # 0 = all
    **COMMON,
)